**1.Import liberaries**

In [1]:
import numpy as np
import pandas as pd
from keras.datasets import mnist

**2.Input dataset.**

In [2]:
(X_train,y_train),(X_test,y_test) = mnist.load_data()

X_train = X_train.reshape(X_train.shape[0],-1) # -1 is input all 28 x 28 =784 in one list.
y_train = y_train.reshape(y_train.shape[0],-1) # here .shape gives error becoze it consider y_train.shape[0] as tuple i.e.
                                               #we use reshape where numpy find this dimension (6000,784) automatically from (6000,28,28).

# Normalize data:
X_train = (X_train - 0)/(255 - 0)  # X_train max =255 pixel.
# initial [0,255] now [0,1]

**3.One hot encoding.**

In [3]:
def encode(y, num_class=10):
  n = y.shape[0]
  ec = np.zeros((n, num_class))
  ec[np.arange(n),y] = 1  # convert handwritten output in binary form.
  return ec

y_train_ohe = encode(y_train)
y_test_ohe = encode(y_test)

**4.Activation Function**

In [4]:
def relu(z):
  return np.maximum(0,z)

def derivative_relu(z):
  return  (z>0).astype(int) #for backpropogation

def softmax(z):
  expz = np.exp(z)
  return expz/np.sum(expz,axis=1,keepdims =True)

**5.Forward**

In [5]:
def foward(X_train,w1,b1,w2,b2):
  z1 = np.dot(X_train,w1) + b1
  a1 = relu(z1)
  z2 = np.dot(a1,w2) + b2
  a2 = softmax(z2)
  return z1,a1,z2,a2


In [6]:
def loss(y_pred,y_train):
  n = X_train.shape[0]
#loss fn and loss variable gives error at last run box.
  Loss = -np.sum(y_train * np.log(y_pred + 1e+1))/n

  return Loss

**6.Backward**

In [7]:
def back_propogation(a2,z1,w2,a1,y_train,X_train):
  n = X_train.shape[0]

  dz2 = a2  - y_train
  dw2 = (1/n)*np.dot(a1.T,dz2)
  db2 = (1/n)* np.sum(dz2,axis=0,keepdims=True)

  da1 = np.dot(dz2,w2.T)
  dz1 = da1 * derivative_relu(z1)
  dw1 = (1/n) * np.dot(X_train.T,dz1)
  db1 = (1/n) * np.sum(dz1,axis=0,keepdims =True)

  return dw1,db1,dw2,db2

**7.Train**

In [8]:
def train(X_train,y_train,epoch=2000,lr=0.01):

  w1 = np.random.rand(784,128)*0.01 # 784- input size, 128- no. of nuerons
  b1= np.random.rand(1,128)
  w2 = np.random.rand(128,10)# 128 - input size(activator), 10 output size(classes)
  b2 = np.random.rand(1,10)

  for i in range(epoch):
    z1,a1,z2,a2 = foward(X_train,w1,b1,w2,b2)
    Loss = loss(y_train,a2)
    #loss fn and loss variable gives error at last run box.

    dw1,db1,dw2,db2 = back_propogation(a2,z1,z2,a1,y_train,X_train)

    w1 -= lr * dw1
    b1 -= lr * db1
    w2 -= lr * dw2
    b2 -= lr * db2

    return w1,b1,w2,b2

**8.predict  and run.**

In [9]:
def predict(X_test,w1,b1,w2,b2):
  z1,a1,z2,a2 = foward(X_test,w1,b1,w2,b2)
  return np.argmax(a2,axis=1)

def accuracy(y_test,y_pred):
  return np.mean(y_test = y_pred)

In [ ]:
w1,b1,w2,b2 = train(X_train,y_train)

y_pred = predict(X_test,w1,b1,w2,b2)
print("Test Accuracy:",accuracy(y_test,y_pred))